# CNN 重要改进架构详解

## 从 AlexNet 到 ConvNeXt 的演进

每一步改进都在解决特定问题：
- 更深导致梯度消失 -> ResNet
- 特征复用 -> DenseNet
- 多尺度特征 -> Inception
- 轻量化 -> MobileNet
- 自动设计 -> EfficientNet
- 现代化 -> ConvNeXt

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. ResNet: 残差网络 (最重要!)

### 问题: 网络越深反而越差?

56层网络的训练误差比20层还高 -- 这不是过拟合，而是退化问题。

### 解决: 残差学习

让网络学习残差 F(x) = H(x) - x, 而不是直接学习 H(x):

```
普通:  output = F(x)
残差:  output = F(x) + x    <- 跳跃连接 (Skip Connection)
```

为什么有效?
- 如果某一层最优是恒等映射, F(x)=0 比 F(x)=x 更容易学习
- 梯度可以直接通过跳跃连接回传, 缓解梯度消失

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        out = self.relu(out)
        return out

block = BasicBlock(64, 64)
x = torch.randn(1, 64, 32, 32)
print(f"BasicBlock: {x.shape} -> {block(x).shape}")
print("输入输出形状相同, 残差连接直接相加")

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += identity
        out = self.relu(out)
        return out

bottleneck = Bottleneck(256, 64)
x = torch.randn(1, 256, 32, 32)
print(f"Bottleneck: {x.shape} -> {bottleneck(x).shape}")
print("1x1降维(256->64) -> 3x3卷积(64) -> 1x1升维(64->256)")
print("减少3x3卷积的通道数, 大幅降低计算量")

In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

def ResNet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

model = ResNet18()
x = torch.randn(1, 1, 28, 28)
print(f"ResNet-18: {x.shape} -> {model(x).shape}")
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 残差连接 vs 无残差连接 对比
class PlainBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = torch.relu(out)
        return out

def build_deep_net(use_residual=True, num_layers=20, channels=16):
    layers = []
    for _ in range(num_layers):
        if use_residual:
            layers.append(BasicBlock(channels, channels))
        else:
            layers.append(PlainBlock(channels))
    return nn.Sequential(
        nn.Conv2d(1, channels, 3, padding=1),
        nn.BatchNorm2d(channels),
        nn.ReLU(),
        *layers,
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(channels, 10),
    )

res_net = build_deep_net(use_residual=True)
plain_net = build_deep_net(use_residual=False)
print(f"残差网络参数量: {sum(p.numel() for p in res_net.parameters()):,}")
print(f"普通网络参数量: {sum(p.numel() for p in plain_net.parameters()):,}")

In [ ]:
torch.manual_seed(42)
X = torch.randn(500, 1, 8, 8)
y = torch.randint(0, 10, (500,))
dataset = torch.utils.data.TensorDataset(X, y)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

def quick_train(model, epochs=50):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for X_batch, y_batch in loader:
            loss = criterion(model(X_batch), y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
    return losses

torch.manual_seed(42)
res_losses = quick_train(build_deep_net(True))
torch.manual_seed(42)
plain_losses = quick_train(build_deep_net(False))

plt.figure(figsize=(8, 5))
plt.plot(plain_losses, label='Plain (No Residual)', color='red')
plt.plot(res_losses, label='ResNet (With Residual)', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Residual Connection Effect (20-layer Network)')
plt.legend()
plt.grid(True)
plt.show()

print(f"普通网络最终 loss: {plain_losses[-1]:.4f}")
print(f"残差网络最终 loss: {res_losses[-1]:.4f}")
print("残差连接使深层网络更容易训练!")

---
## 2. DenseNet: 密集连接网络

ResNet 的跳跃连接是跨层相加, DenseNet 改为跨层拼接:

```
ResNet:   output = F(x) + x          (相加, 维度不变)
DenseNet: output = concat(F(x), x)    (拼接, 维度增加)
```

每一层的输入包含之前所有层的输出, 实现特征复用。

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, 4 * growth_rate, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(4 * growth_rate)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(4 * growth_rate, growth_rate, 3, padding=1, bias=False)

    def forward(self, x):
        new_features = self.conv1(self.relu1(self.bn1(x)))
        new_features = self.conv2(self.relu2(self.bn2(new_features)))
        return torch.cat([x, new_features], dim=1)

layer = DenseLayer(in_channels=64, growth_rate=32)
x = torch.randn(1, 64, 8, 8)
out = layer(x)
print(f"DenseLayer: {x.shape} -> {out.shape}")
print(f"通道数: 64 -> 64+32=96 (拼接增加了 growth_rate 个通道)")

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super().__init__()
        self.layers = nn.ModuleList()
        channels = in_channels
        for i in range(num_layers):
            self.layers.append(DenseLayer(channels, growth_rate))
            channels += growth_rate

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.pool = nn.AvgPool2d(2, 2)

    def forward(self, x):
        x = self.conv(self.relu(self.bn(x)))
        x = self.pool(x)
        return x

block = DenseBlock(num_layers=4, in_channels=64, growth_rate=32)
x = torch.randn(1, 64, 8, 8)
out = block(x)
print(f"DenseBlock (4层): {x.shape} -> {out.shape}")
print(f"通道数: 64 -> 64+4*32=192")
print("每层的输出都被后续所有层使用 -> 特征复用")

trans = TransitionLayer(192, 96)
out2 = trans(out)
print(f"\nTransitionLayer: {out.shape} -> {out2.shape}")
print("1x1卷积压缩通道 + 2x2平均池化降低尺寸")

In [ ]:
print("=== DenseNet vs ResNet ===")
print()
print("ResNet:")
print("  连接方式: 相加 (F(x) + x)")
print("  特征流动: 只跨越一层")
print("  通道数:   不变")
print("  优点:     简单高效")
print()
print("DenseNet:")
print("  连接方式: 拼接 (concat(F(x), x))")
print("  特征流动: 每层与所有前层连接")
print("  通道数:   递增 (需要过渡层压缩)")
print("  优点:     特征复用, 参数更少")
print("  缺点:     显存消耗大 (存储所有中间特征)")

---
## 3. Inception: 多尺度特征提取

不同大小的卷积核捕捉不同尺度的特征, 与其选择一种, 不如并行使用多种, 让网络自己选择。

In [ ]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3_reduce, ch3x3, ch5x5_reduce, ch5x5, pool_proj):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, 1),
            nn.BatchNorm2d(ch1x1),
            nn.ReLU(inplace=True),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3_reduce, 1),
            nn.BatchNorm2d(ch3x3_reduce),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch3x3_reduce, ch3x3, 3, padding=1),
            nn.BatchNorm2d(ch3x3),
            nn.ReLU(inplace=True),
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5_reduce, 1),
            nn.BatchNorm2d(ch5x5_reduce),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch5x5_reduce, ch5x5, 3, padding=2),
            nn.BatchNorm2d(ch5x5),
            nn.ReLU(inplace=True),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, 1),
            nn.BatchNorm2d(pool_proj),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)

inception = InceptionModule(64, 16, 24, 32, 4, 8, 16)
x = torch.randn(1, 64, 28, 28)
out = inception(x)
total_ch = 16 + 32 + 8 + 16
print(f"Inception: {x.shape} -> {out.shape}")
print(f"4个分支输出通道: 16 + 32 + 8 + 16 = {total_ch}")
print()
print("4个并行分支:")
print("  分支1: 1x1 卷积 (捕捉点状特征)")
print("  分支2: 1x1 -> 3x3 (捕捉局部特征)")
print("  分支3: 1x1 -> 5x5 (捕捉更大范围特征)")
print("  分支4: 池化 -> 1x1 (捕捉全局特征)")

In [ ]:
print("=== 1x1 卷积降维的作用 ===")
print()
print("直接用 3x3 卷积:")
print("  输入256通道 -> 3x3卷积(256, 64) -> 输出64通道")
params_direct = 256 * 64 * 3 * 3
print(f"  参数量: {params_direct:,}")
print()
print("先用 1x1 降维再用 3x3:")
print("  输入256通道 -> 1x1(256,32) -> 3x3(32,64) -> 输出64通道")
params_reduce = 256 * 32 * 1 * 1 + 32 * 64 * 3 * 3
print(f"  参数量: {params_reduce:,}")
print(f"  减少: {params_direct / params_reduce:.1f}x")
print()
print("1x1 卷积降维大幅减少计算量, 这是 Inception 的关键创新")

---
## 4. MobileNet: 轻量化网络

将标准卷积分解为:
1. 深度卷积 (Depthwise Conv): 每个通道单独卷积
2. 逐点卷积 (Pointwise Conv): 1x1 卷积融合通道

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, 3,
            stride=stride, padding=1, groups=in_channels, bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.depthwise(x)))
        x = self.relu(self.bn2(self.pointwise(x)))
        return x

in_ch, out_ch = 64, 128
std_conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
ds_conv = DepthwiseSeparableConv(in_ch, out_ch)

std_params = sum(p.numel() for p in std_conv.parameters())
ds_params = sum(p.numel() for p in ds_conv.parameters())

print(f"标准卷积参数量:       {std_params:,}")
print(f"深度可分离卷积参数量: {ds_params:,}")
print(f"参数减少: {std_params / ds_params:.1f}x")

In [ ]:
class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()
        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True),
            ])
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride=stride,
                      padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),
        ])
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        return self.conv(x)

block = InvertedResidual(64, 128, stride=1, expand_ratio=6)
x = torch.randn(1, 64, 16, 16)
print(f"InvertedResidual: {x.shape} -> {block(x).shape}")
print()
print("MobileNet V2 倒残差结构:")
print("  ResNet:   降维 -> 3x3 -> 升维   (沙漏形)")
print("  MobileV2: 升维 -> 3x3 -> 降维   (纺锤形)")
print("  最后的1x1不加ReLU (线性瓶颈), 保留信息")

---
## 5. SE Block: 通道注意力

SE (Squeeze-and-Excitation) 让网络自动学习哪些通道更重要。
这是 EfficientNet 和很多现代网络的核心组件。

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.squeeze(x).view(b, c)
        s = self.excitation(s).view(b, c, 1, 1)
        return x * s

se = SEBlock(64)
x = torch.randn(2, 64, 8, 8)
out = se(x)
print(f"SE Block: {x.shape} -> {out.shape}")
print()
print("SE Block (通道注意力):")
print("  Squeeze:    全局池化 -> 每个通道一个数")
print("  Excitation: FC -> ReLU -> FC -> Sigmoid -> 通道权重")
print("  Scale:      原特征 * 通道权重")
print("  效果:       让网络自动学习哪些通道更重要")

---
## 6. EfficientNet: 复合缩放

之前的网络缩放只调整一个维度 (深度/宽度/分辨率),
EfficientNet 提出复合缩放: 用一组系数同时缩放三个维度。

```
depth:      d = alpha^phi
width:      w = beta^phi
resolution: r = gamma^phi
约束: alpha * beta^2 * gamma^2 约等于 2
```

phi 是用户指定的缩放系数, alpha/beta/gamma 通过搜索确定。
EfficientNet-B0 到 B7, phi 从 0 递增, 模型越来越大。

In [ ]:
# 缩放策略可视化
methods = {
    'Baseline': (1.0, 1.0, 1.0),
    'Scale Depth': (2.0, 1.0, 1.0),
    'Scale Width': (1.0, 2.0, 1.0),
    'Scale Resolution': (1.0, 1.0, 2.0),
    'Compound': (1.3, 1.2, 1.15),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, (name, (d, w, r)) in zip(axes, methods.items()):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    layers = int(4 * d)
    ch = int(16 * w)
    for i in range(layers):
        y = 1 + i * (8 / max(layers, 1))
        rect = plt.Rectangle((1, y), min(ch / 16 * 3, 8), min(r * 1.5, 3),
                            facecolor='steelblue', alpha=0.7, edgecolor='black')
        ax.add_patch(rect)
    ax.set_title(f'{name}\nd={d:.1f}, w={w:.1f}, r={r:.2f}', fontsize=9)
    ax.axis('off')

plt.suptitle('EfficientNet Scaling Strategies', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("复合缩放同时调整深度、宽度、分辨率, 效果最好")

---
## 7. ConvNeXt: 现代化 CNN

2022年的工作, 将 Transformer 的设计理念迁移回 CNN, 纯 CNN 达到 Swin Transformer 的水平。

### 从 ResNet 到 ConvNeXt 的改进

| 改进 | ResNet | ConvNeXt |
|------|--------|----------|
| 卷积核 | 3x3 | 7x7 深度卷积 |
| 激活函数 | ReLU | GELU |
| 归一化 | BatchNorm | LayerNorm |
| 注意力 | 无 | SE 通道注意力 |
| 位置 | Conv->BN->ReLU | Conv->LN->GELU |

In [ ]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = x.permute(0, 3, 1, 2)
        return x + residual

block = ConvNeXtBlock(96)
x = torch.randn(1, 96, 14, 14)
print(f"ConvNeXt Block: {x.shape} -> {block(x).shape}")
print()
print("ConvNeXt 的关键改变:")
print("  7x7 深度卷积 (更大感受野)")
print("  LayerNorm 替代 BatchNorm")
print("  GELU 替代 ReLU")
print("  用 Linear 替代 1x1 Conv (等价但更灵活)")

---
## 8. 架构对比总结

In [ ]:
print("=== CNN 架构演进对比 ===")
print()
print(f"{'模型':<20} {'年份':<6} {'参数量':<15} {'核心创新':<30}")
print("-" * 75)
print(f"{'AlexNet':<20} {'2012':<6} {'60M':<15} {'ReLU + Dropout + GPU':<30}")
print(f"{'VGGNet':<20} {'2014':<6} {'138M':<15} {'3x3卷积堆叠':<30}")
print(f"{'Inception':<20} {'2014':<6} {'6.8M':<15} {'多尺度并行 + 1x1降维':<30}")
print(f"{'ResNet':<20} {'2015':<6} {'25.6M':<15} {'残差连接':<30}")
print(f"{'DenseNet':<20} {'2017':<6} {'8.0M':<15} {'密集连接 + 特征复用':<30}")
print(f"{'MobileNet V2':<20} {'2018':<6} {'3.5M':<15} {'深度可分离 + 倒残差':<30}")
print(f"{'EfficientNet':<20} {'2019':<6} {'5.3M-66M':<15} {'复合缩放 + SE':<30}")
print(f"{'ConvNeXt':<20} {'2022':<6} {'28M-350M':<15} {'Transformer化CNN':<30}")

In [ ]:
print("=== 如何选择 CNN 架构? ===")
print()
print("1. 通用分类/检测/分割:")
print("   首选: ResNet-50 (简单可靠)")
print("   进阶: ConvNeXt (SOTA 性能)")
print()
print("2. 移动端/嵌入式:")
print("   首选: MobileNet V2/V3")
print("   进阶: EfficientNet-Lite")
print()
print("3. 追求最高精度:")
print("   首选: EfficientNet-V2")
print("   进阶: ConvNeXt-XL")
print()
print("4. 参数受限:")
print("   首选: MobileNet V3 Small")
print("   进阶: EfficientNet-B0")
print()
print("重要: 架构选择不如数据和训练策略重要!")
print("ResNet-50 + 好的数据增强 往往比 复杂架构 + 差数据 效果好")

In [ ]:
print("=== 核心设计模式总结 ===")
print()
print("1. 残差连接 (ResNet)")
print("   output = F(x) + x")
print("   -> 解决退化问题, 使极深网络可训练")
print()
print("2. 密集连接 (DenseNet)")
print("   output = concat(x, F1(x), F2(x), ...)")
print("   -> 特征复用, 参数更少")
print()
print("3. 多尺度并行 (Inception)")
print("   output = concat(1x1(x), 3x3(x), 5x5(x), pool(x))")
print("   -> 同时捕捉不同尺度特征")
print()
print("4. 深度可分离卷积 (MobileNet)")
print("   Depthwise(x) -> Pointwise(x)")
print("   -> 大幅减少计算量")
print()
print("5. 通道注意力 (SE Block)")
print("   x * sigmoid(FC(AvgPool(x)))")
print("   -> 自适应调整通道重要性")
print()
print("6. 复合缩放 (EfficientNet)")
print("   同时缩放 depth + width + resolution")
print("   -> 均衡扩展, 效率最优")
print()
print("7. 现代化改造 (ConvNeXt)")
print("   大核深度卷积 + LayerNorm + GELU")
print("   -> 吸收 Transformer 优势, 纯 CNN 复兴")

---
## 总结

### 架构演进的核心逻辑

```
问题: 网络太浅 -> 表达力不够
  -> 加深
问题: 太深 -> 梯度消失/退化
  -> ResNet 残差连接
问题: 参数太多 -> 计算量大
  -> 1x1降维(Inception) / 深度可分离(MobileNet)
问题: 缩放不均衡
  -> EfficientNet 复合缩放
问题: Transformer 更强?
  -> ConvNeXt 现代化 CNN
```

### 关键教训

1. **残差连接是最重要的创新**, 几乎所有现代网络都在用
2. **1x1 卷积**是控制计算量的利器
3. **注意力机制**(SE Block)让网络学会看哪里重要
4. **架构选择不如训练策略重要**
5. **实际项目建议从 ResNet-50 开始**, 简单可靠